# Unidad 42. Comparaciones categóricas

**Bloque:** Seaborn (UT3) · **Tiempo orientativo:** 5–7 horas

> Las categorías organizan observaciones en grupos. Un gráfico categórico debe dejar claro si compara el número de casos, un resumen de una medida o la distribución completa dentro de cada grupo.

## Resultados de aprendizaje

Al terminar podrás distinguir `countplot`, `barplot`, `boxplot`, `violinplot`, `stripplot`, `swarmplot` y `catplot`; elegir una visualización según la pregunta; y explicar qué información se pierde al resumir un grupo con una barra.

**Prerequisitos:** unidades 39–41 y operaciones básicas de agrupación de Pandas.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")


def ruta_datos(nombre):
    for ruta in [Path("datos") / "seaborn" / nombre, Path("..") / "datos" / "seaborn" / nombre]:
        if ruta.exists():
            return ruta
    raise FileNotFoundError(f"No se encontró {nombre}. Abre el cuaderno desde la carpeta del curso.")

evaluaciones = pd.read_csv(ruta_datos("42_evaluaciones_configuraciones.csv"))
evaluaciones.head()

## 1. Primero, cuenta o mide

La pregunta **«¿Cuántas observaciones tiene cada configuración?»** se responde con un conteo. La pregunta **«¿Qué exactitud presentan las observaciones de cada configuración?»** necesita una medida numérica y no debe confundirse con el tamaño del grupo.

`countplot` cuenta filas. No debe usarse para representar la media de una columna numérica.

In [ ]:
print(evaluaciones.groupby(["configuracion", "turno"]).size())

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=evaluaciones, x="configuracion", hue="turno", ax=ax)
ax.set(title="Número de evaluaciones simuladas", xlabel="Configuración", ylabel="Número de observaciones")
ax.legend(title="Turno")
plt.show()

Los grupos tienen el mismo número de observaciones, algo que debes comprobar antes de comparar resúmenes.

## 2. Barras: una estimación, no todos los valores

`barplot` resume una medida numérica por categoría. Por defecto estima la media y dibuja un intervalo de incertidumbre calculado mediante remuestreo. La barra no muestra la dispersión individual ni demuestra una diferencia real entre grupos. Para evitar resultados aleatorios en el ejemplo, fijamos `seed`.

Si el objetivo es mostrar todos los valores, una barra es insuficiente: usa puntos, caja o violín según el tamaño de la muestra y la pregunta.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=evaluaciones, x="configuracion", y="exactitud_pct", hue="turno", errorbar=("ci", 90), seed=2026, ax=ax)
ax.set(title="Exactitud media simulada por grupo", xlabel="Configuración", ylabel="Exactitud (%)")
ax.legend(title="Turno")
plt.show()

## 3. Distribución dentro de los grupos

Un `boxplot` muestra mediana, cuartiles y posibles valores alejados según una regla convencional. Un `violinplot` añade una forma suavizada de densidad: puede ser expresivo, pero con muestras pequeñas no debe interpretarse como una forma precisa. `stripplot` dibuja cada punto; `swarmplot` intenta evitar solapamientos, aunque puede ser lento o difícil de leer con muchos datos.

En esta tabla pequeña, combinar una caja con puntos hace visible tanto el resumen como las observaciones.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=evaluaciones, x="configuracion", y="exactitud_pct", hue="turno", palette="pastel", ax=ax)
sns.stripplot(data=evaluaciones, x="configuracion", y="exactitud_pct", hue="turno", dodge=True, color="#333333", alpha=0.7, ax=ax)
ax.set(title="Exactitudes individuales y resumen por grupo", xlabel="Configuración", ylabel="Exactitud (%)")
# La superposición crea dos leyendas; conservamos solo la primera.
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="Turno")
plt.show()

## 4. `catplot`: una figura completa

`catplot` es la interfaz de nivel de figura para gráficos categóricos. Permite elegir `kind="box"`, `"violin"`, `"strip"`, `"swarm"`, `"bar"` o `"count"`, y más adelante dividir los datos en paneles. Igual que `relplot`, crea su propia figura.

No fijes límites verticales estrechos solo para que diferencias pequeñas parezcan grandes. En porcentajes, un eje que empieza en cero puede facilitar una comparación honesta, aunque el rango elegido siempre debe explicarse.

In [ ]:
grafico_categorico = sns.catplot(data=evaluaciones, x="configuracion", y="tiempo_ms", kind="violin", hue="turno", inner="quartile", height=4, aspect=1.4)
grafico_categorico.set_axis_labels("Configuración", "Tiempo de respuesta (ms)")
grafico_categorico.figure.suptitle("Distribución simulada de tiempos", y=1.03)
plt.show()

## Práctica guiada

Crea un `swarmplot` de `tiempo_ms` por configuración. Añade `hue="turno"`, `dodge=True`, título, unidades y leyenda. Comprueba que se ven las 36 observaciones. Explica en un comentario por qué un swarmplot puede dejar de ser adecuado si hay miles de puntos.

In [ ]:
# Completa la práctica guiada aquí.
assert len(evaluaciones) == 36, f"Se esperaban 36 observaciones, pero hay {len(evaluaciones)}"

fig, ax = plt.subplots(figsize=(7.5, 4.2))
sns.swarmplot(
    data=evaluaciones,
    x="configuracion",
    y="tiempo_ms",
    hue="turno",
    dodge=True,
    size=7,
    ax=ax,
)
ax.set(
    title="Tiempos de respuesta individuales por configuración y turno",
    xlabel="Configuración",
    ylabel="Tiempo de respuesta (ms)",
)
ax.legend(title="Turno")
plt.show()

# Explicación: Con miles de puntos, swarmplot se vuelve muy lento porque calcula desplazamientos
# laterales para evitar que los puntos se toquen; además, cuando el ancho de la columna se agota,
# los puntos se apilan o se salen del espacio asignado volviéndose ilegibles.

- **[U42.3]** `assert len(evaluaciones) == 36, ...` verifica explícitamente que la tabla contiene las 36 observaciones individuales antes de dibujarlas,
- **[U42.3]** `sns.swarmplot(data=evaluaciones, x="configuracion", y="tiempo_ms", hue="turno", dodge=True, size=7, ax=ax)` dibuja en nivel de ejes cada observación individual desplazando lateralmente los puntos para evitar solapamientos y separando los subgrupos de `turno` (`dodge=True`),
- **[U42.3]** `ax.set(...)`, `ax.legend(title="Turno")` y `plt.show()` añaden título, unidades (`ms`), leyenda y renderizan la gráfica.

## Ejercicios

### Ejercicio 42.01 · Conteos verificables

Construye una tabla de conteos por configuración y turno. Comprueba que cada combinación tiene seis filas y represéntala con `countplot`.

### Ejercicio 42.02 · Media y puntos

Representa la exactitud por configuración con un `barplot` y superpone un `stripplot`. Escribe qué aporta cada capa y por qué la barra no basta.

### Ejercicio 42.03 · Dos resúmenes distintos

Crea un `boxplot` y un `violinplot` de `tiempo_ms` por configuración. Indica una diferencia entre la información que ambos comunican y una precaución al interpretar el violín.

### Ejercicio 42.04 · Figura de nivel de figura

Crea un `catplot(kind="box")` de exactitud por configuración, con paneles por turno (`col="turno"`). Guarda el resultado en `cajas_por_turno` y etiqueta ejes y título.

### Ejercicio 42.05 · Miniproyecto: equilibrio entre medida y tiempo

Elabora una figura o conjunto breve de dos figuras para comparar exactitud y tiempo de respuesta de las configuraciones simuladas. Incluye los puntos individuales en al menos una figura. Redacta tres conclusiones: pauta observada, limitación y dato adicional necesario antes de elegir una configuración real.

In [ ]:
# ==========================================
# Ejercicio 42.01 · Conteos verificables
# ==========================================
tabla_conteos = (
    evaluaciones.groupby(["configuracion", "turno"], as_index=False)
    .size()
    .rename(columns={"size": "n_observaciones"})
)
display(tabla_conteos)

assert (tabla_conteos["n_observaciones"] == 6).all(), (
    "No todas las combinaciones de configuración y turno tienen exactamente 6 filas"
)

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=evaluaciones, x="configuracion", hue="turno", ax=ax)
ax.set(
    title="42.01 · Número de evaluaciones por configuración y turno",
    xlabel="Configuración",
    ylabel="Número de observaciones",
)
ax.legend(title="Turno")
plt.show()

# ==========================================
# Ejercicio 42.02 · Media y puntos
# ==========================================
fig, ax = plt.subplots(figsize=(7.5, 4.2))
sns.barplot(
    data=evaluaciones,
    x="configuracion",
    y="exactitud_pct",
    errorbar=("ci", 90),
    seed=2026,
    color="#88CCEE",
    alpha=0.75,
    ax=ax,
)
sns.stripplot(
    data=evaluaciones,
    x="configuracion",
    y="exactitud_pct",
    color="#222222",
    jitter=0.15,
    size=6.5,
    alpha=0.85,
    ax=ax,
)
ax.set(
    title="42.02 · Exactitud media (IC 90 %) y observaciones individuales",
    xlabel="Configuración",
    ylabel="Exactitud (%)",
)
plt.show()
# Qué aporta cada capa: barplot estima la media y muestra su intervalo de confianza (90 %), mientras
# que stripplot conserva cada observación individual. La barra por sí sola no basta porque oculta
# la dispersión interna, la asimetría, los posibles valores atípicos y el tamaño real de cada grupo.

# ==========================================
# Ejercicio 42.03 · Dos resúmenes distintos
# ==========================================
fig, ejes = plt.subplots(1, 2, sharey=True, figsize=(11, 4.2), layout="constrained")
sns.boxplot(data=evaluaciones, x="configuracion", y="tiempo_ms", palette="pastel", ax=ejes[0])
ejes[0].set(
    title="Boxplot: mediana, cuartiles y atípicos",
    xlabel="Configuración",
    ylabel="Tiempo de respuesta (ms)",
)

sns.violinplot(
    data=evaluaciones,
    x="configuracion",
    y="tiempo_ms",
    palette="pastel",
    inner="quartile",
    ax=ejes[1],
)
ejes[1].set(
    title="Violinplot: densidad suavizada y cuartiles",
    xlabel="Configuración",
    ylabel="Tiempo de respuesta (ms)",
)
fig.suptitle("42.03 · Comparación de tiempo de respuesta por configuración")
plt.show()
# Diferencia y precaución: El boxplot resume estrictamente estadísticos de posición (mediana, IQR y
# posibles atípicos), mientras que el violinplot añade la estimación suavizada de densidad (KDE).
# Precaución: con muestras pequeñas (12 puntos por grupo), la forma suavizada del violín puede
# sugerir colas o multimodalidades que los datos no sostienen.

# ==========================================
# Ejercicio 42.04 · Figura de nivel de figura
# ==========================================
cajas_por_turno = sns.catplot(
    data=evaluaciones,
    x="configuracion",
    y="exactitud_pct",
    col="turno",
    kind="box",
    palette="pastel",
    height=4,
    aspect=1.15,
)
cajas_por_turno.set_axis_labels("Configuración", "Exactitud (%)")
cajas_por_turno.set_titles("Turno: {col_name}")
cajas_por_turno.figure.suptitle("42.04 · Distribución de exactitud por configuración y turno", y=1.04)
plt.show()

# ==========================================
# Ejercicio 42.05 · Miniproyecto: equilibrio entre medida y tiempo
# ==========================================
fig, ejes = plt.subplots(1, 2, figsize=(12.5, 4.5), layout="constrained")

sns.boxplot(
    data=evaluaciones,
    x="configuracion",
    y="exactitud_pct",
    hue="turno",
    palette="pastel",
    fliersize=0,
    ax=ejes[0],
)
sns.stripplot(
    data=evaluaciones,
    x="configuracion",
    y="exactitud_pct",
    hue="turno",
    dodge=True,
    palette=["#222222", "#222222"],
    alpha=0.8,
    ax=ejes[0],
)
handles, labels = ejes[0].get_legend_handles_labels()
ejes[0].legend(handles[:2], labels[:2], title="Turno")
ejes[0].set(
    title="1. Exactitud por configuración y turno",
    xlabel="Configuración",
    ylabel="Exactitud (%)",
)

sns.boxplot(
    data=evaluaciones,
    x="configuracion",
    y="tiempo_ms",
    hue="turno",
    palette="pastel",
    fliersize=0,
    ax=ejes[1],
)
sns.stripplot(
    data=evaluaciones,
    x="configuracion",
    y="tiempo_ms",
    hue="turno",
    dodge=True,
    palette=["#222222", "#222222"],
    alpha=0.8,
    ax=ejes[1],
)
handles2, labels2 = ejes[1].get_legend_handles_labels()
ejes[1].legend(handles2[:2], labels2[:2], title="Turno")
ejes[1].set(
    title="2. Tiempo de respuesta por configuración y turno",
    xlabel="Configuración",
    ylabel="Tiempo de respuesta (ms)",
)

fig.suptitle("42.05 · Equilibrio entre exactitud y tiempo de respuesta (datos simulados)")
plt.show()

### Explicación línea por línea (Ejercicios 42.01 a 42.05)

- **[U42.1]** `tabla_conteos = evaluaciones.groupby(["configuracion", "turno"], as_index=False).size().rename(columns={"size": "n_observaciones"})` agrupa por `configuracion` y `turno` y cuenta el número de filas de cada subgrupo[cite: 5],
- **[U42.1]** `assert (tabla_conteos["n_observaciones"] == 6).all(), ...` valida mediante aserción que todas las combinaciones están equilibradas con exactamente 6 registros[cite: 5],
- **[U42.1]** `sns.countplot(data=evaluaciones, x="configuracion", hue="turno", ax=ax)` representa el conteo de filas por categoría (nunca medias numéricas) en nivel de ejes[cite: 5],
- **[U42.2]** `sns.barplot(data=evaluaciones, x="configuracion", y="exactitud_pct", errorbar=("ci", 90), seed=2026, color="#88CCEE", alpha=0.75, ax=ax)` estima la media de `exactitud_pct` por configuración junto con su intervalo de incertidumbre del 90 % fijando `seed=2026` para reproducibilidad[cite: 5],
- **[U42.2 / U42.3]** `sns.stripplot(data=evaluaciones, x="configuracion", y="exactitud_pct", color="#222222", jitter=0.15, size=6.5, alpha=0.85, ax=ax)` superpone todos los puntos individuales con un ligero desplazamiento horizontal aleatorio (`jitter=0.15`) para recuperar la dispersión que la barra oculta[cite: 5],
- **[U42.3]** `fig, ejes = plt.subplots(1, 2, sharey=True, figsize=(11, 4.2), layout="constrained")` crea una cuadrícula de dos paneles con escala vertical compartida (`sharey=True`) para comparar directamente los tiempos[cite: 5],
- **[U42.3]** `sns.boxplot(data=evaluaciones, x="configuracion", y="tiempo_ms", palette="pastel", ax=ejes[0])` dibuja en el panel izquierdo la mediana, el rango intercuartílico y los bigotes[cite: 5],
- **[U42.3]** `sns.violinplot(data=evaluaciones, x="configuracion", y="tiempo_ms", palette="pastel", inner="quartile", ax=ejes[1])` dibuja en el panel derecho la curva suavizada de densidad junto con las líneas interiores de cuartiles (`inner="quartile"`), recordando que en muestras pequeñas puede suavizar colas inexistentes[cite: 5],
- **[U42.4]** `cajas_por_turno = sns.catplot(data=evaluaciones, x="configuracion", y="exactitud_pct", col="turno", kind="box", palette="pastel", height=4, aspect=1.15)` crea una figura completa de **nivel de figura** (`FacetGrid`) con un panel de diagrama de cajas (`kind="box"`) por cada categoría de `turno` (`col="turno"`)[cite: 5],
- **[U42.4]** `cajas_por_turno.set_axis_labels("Configuración", "Exactitud (%)")`, `cajas_por_turno.set_titles("Turno: {col_name}")` y `cajas_por_turno.figure.suptitle(..., y=1.04)` configuran las etiquetas exteriores, los títulos de panel y el encabezado global del objeto de nivel de figura[cite: 5],
- **[U42.3]** `sns.boxplot(..., fliersize=0, ax=ejes[0])` y `sns.stripplot(..., dodge=True, palette=["#222222", "#222222"], alpha=0.8, ax=ejes[0])` combinan en el miniproyecto final el resumen por cuartiles con las 36 observaciones individuales alineadas por subgrupo sin duplicar puntos atípicos ni generar advertencias de paleta[cite: 5],
- **[U42.3]** `handles, labels = ejes[0].get_legend_handles_labels()` y `ejes[0].legend(handles[:2], labels[:2], title="Turno")` eliminan las entradas duplicadas de la leyenda generadas al superponer dos capas en los mismos ejes[cite: 5].

---

## Autoevaluación y cierre

1. ¿Qué representa un `countplot`?
2. ¿Qué representa normalmente la altura de un `barplot`?
3. ¿Qué información recuperas al añadir puntos a un boxplot o a una barra?
4. ¿Qué diferencia práctica hay entre `boxplot` y `violinplot`?
5. ¿Cuándo usarías `catplot` en lugar de un gráfico sobre `ax`?

La siguiente unidad conectará tablas pivote, correlaciones y mapas de calor.